# 03 · Compare hypothetical growth approaches

The task is to compare testable approaches, not estimate a market or predict uplift. **All counts and shares below are SYNTHETIC:** illustrative results on independently generated synthetic data; not findings from the ADK consumer survey.

This independent notebook generates its own fixture. Thresholds are transparent editorial assumptions; the source data contain no outcomes that can validate the approaches. See [the strategy case](../docs/STRATEGY_CASE.md).

In [ ]:
import pandas as pd
from IPython.display import display

from retail_case.schema import SYNTHETIC_NOTICE
from retail_case.synthetic import generate_data

df = generate_data(n=720, seed=2026, missing_rate=0.03)


def show_synthetic_table(title, table):
    print(f"SYNTHETIC — {title}")
    print(SYNTHETIC_NOTICE)
    display(table)

In [ ]:
from retail_case.validation import validate_data

validate_data(df)
# Each rule has its own observable denominator. Counts may overlap.
rule_specs = [
    (
        "Broad meal-occasion comparison / no segmentation",
        [],
        lambda data: pd.Series(True, index=data.index),
    ),
    (
        "Low store activity with cross-channel activity",
        ["monthly_store_visits", "weekly_other_channel_visits"],
        lambda data: (data.monthly_store_visits <= 2) & (data.weekly_other_channel_visits >= 2),
    ),
    (
        "Time-pressured convenience comparison",
        ["shopping_time_pressure", "convenience_preference"],
        lambda data: (data.shopping_time_pressure >= 4) & (data.convenience_preference >= 4),
    ),
    (
        "Meal-planning exploration",
        ["meal_planning_interest", "discovery_interest"],
        lambda data: (data.meal_planning_interest >= 4) & (data.discovery_interest >= 4),
    ),
]
opportunity_rows = []
for approach, required, rule in rule_specs:
    observable = df.dropna(subset=required) if required else df
    mask = rule(observable)
    opportunity_rows.append(
        {
            "hypothetical_approach": approach,
            "observable_rows": len(observable),
            "rule_matches": int(mask.sum()),
            "share_of_observable_fixture": float(mask.mean()),
            "rows_with_missing_required_fields": len(df) - len(observable),
        }
    )
show_synthetic_table(
    "Conditional rule coverage; not demand, market size, or predicted response",
    pd.DataFrame(opportunity_rows),
)

In [ ]:
evidence_gaps = pd.DataFrame(
    [
        {
            "status": "ASSUMPTION",
            "approach": "Broad comparison",
            "proposed_mechanism": "Reduce effort comparing ways to complete a meal",
            "missing_evidence": (
                "Usability and whether comparison adds value beyond the existing floor"
            ),
        },
        {
            "status": "INTERPRETATION",
            "approach": "Cross-channel link",
            "proposed_mechanism": "Offer a reason to include the store in a meal-shopping route",
            "missing_evidence": "Actual visit motives, substitution, and repeated visits",
        },
        {
            "status": "RECOMMENDATION",
            "approach": "Convenience comparison",
            "proposed_mechanism": "Show prepared, frozen, ingredient, and mixed options together",
            "missing_evidence": "Whether the comparison is useful under time pressure",
        },
        {
            "status": "UNKNOWN",
            "approach": "Meal-planning exploration",
            "proposed_mechanism": "Use changing meal themes to support discovery",
            "missing_evidence": "Real incremental interaction and demand after novelty fades",
        },
    ]
)
print("HYPOTHETICAL CASE — qualitative opportunity mechanisms and evidence gaps")
display(evidence_gaps)

## Conditional next step

Rule coverage checks that an analysis can produce explicit denominators. It supplies no basis for favoring an approach by expected commercial return. Overlapping rows must not be added into a total addressable market, and no conversion or sales assumptions are inserted.

**Recommendation:** retain a broad, unsegmented comparison experience as a baseline and test task usefulness before choosing a narrower audience. [04 · Segmentation](04_segmentation.ipynb) asks whether exploratory partitions add a descriptive view, while preserving the no-segmentation alternative.